# Frozen-Claim Large-Model Scale Replication

This confirmatory notebook tests whether the representation → composition → answer-use pathway found in Qwen3.5-9B persists or reorganizes at 27B scale.

[实验设置与运行说明](../../../docs/experiment_notes/09_appendix_factor_routes/3704d9ff58b16dd3.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Setup


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'hf_xet>=1.1', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the frozen-claim scale protocol
cell_stage('Freezing model revision, manifest, layer map, and decision branches')
import hashlib
import json
from datetime import datetime, timezone

MODEL_KEY = 'qwen27b'
MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_SLUG = 'Qwen__Qwen3.5-27B__373612cb'
REFERENCE_MODEL_KEY = 'qwen9b'
REFERENCE_MODEL_ID = 'Qwen/Qwen3.5-9B'

REFERENCE_LAYER_COUNT = 32
EXPECTED_LAYER_COUNT = 64
REFERENCE_SITES = (
    (16, 'all_text_except_final', 'middle_text'),
    (16, 'options_tokens', 'middle_options'),
    (27, 'final_prompt_token', 'late_answer'),
)

def map_layer_by_depth(layer_index, source_layers, target_layers):
    depth = float(layer_index) / float(source_layers - 1)
    return int(round(depth * (target_layers - 1)))

FROZEN_SITES = tuple(
    (
        map_layer_by_depth(layer, REFERENCE_LAYER_COUNT, EXPECTED_LAYER_COUNT),
        scope, label,
    )
    for layer, scope, label in REFERENCE_SITES
)
assert FROZEN_SITES == (
    (33, 'all_text_except_final', 'middle_text'),
    (33, 'options_tokens', 'middle_options'),
    (55, 'final_prompt_token', 'late_answer'),
), FROZEN_SITES
SITE_BY_LABEL = {
    label: (layer, scope) for layer, scope, label in FROZEN_SITES
}
CAPTURE_LAYERS = tuple(sorted({layer for layer, _, _ in FROZEN_SITES}))

MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
SEED = 20260907
EXPECTED_TRIPLETS = 48
TASKS = (
    'side_units', 'side_distance', 'side_torque',
    'net_torque', 'outcome',
)
DONOR_TYPES = ('weight_source', 'distance_source')
PATCH_CONDITION_BATCH_SIZE = 1
NATURAL_EFFECT_EPS = 0.02
EQUIVALENCE_FRACTION = 0.25
EQUIVALENCE_FRACTIONS = (0.20, 0.25, 0.30, 0.40)
CROSS_SCALE_EQUIVALENCE_BOUND = 0.25
BOOTSTRAP_REPLICATES = 20000
SIGN_FLIP_REPLICATES = 20000
ALPHA = 0.05
FIXED_VARIANT = 'permutation_0'
IMAGE_TOKEN_ID_OVERRIDE = None
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.02

SOURCE_ROOT = (
    PROJECT_ROOT / 'results' / 'orthogonal_value_binding_control_v1'
)
MANIFEST_PATH = SOURCE_ROOT / 'orthogonal_triplet_manifest.jsonl'
MANIFEST_LOCK_PATH = SOURCE_ROOT / 'manifest_lock.json'
REFERENCE_ROOT = SOURCE_ROOT / 'models' / 'Qwen__Qwen3.5-9B'
REFERENCE_PAIRED_PATH = (
    REFERENCE_ROOT / 'analysis' / 'paired_triplet_metrics.csv'
)
REFERENCE_DECISION_PATH = (
    REFERENCE_ROOT / 'analysis' / 'orthogonal_value_binding_decision.json'
)

EXPERIMENT_ROOT = (
    PROJECT_ROOT / 'results' / 'large_model_frozen_claim_replication_v1'
)
OUTPUT_ROOT = EXPERIMENT_ROOT / 'models' / MODEL_SLUG
SHARD_DIR = OUTPUT_ROOT / 'triplet_shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
COMPARISON_DIR = EXPERIMENT_ROOT / 'scale_comparison'
for path in (
    EXPERIMENT_ROOT, OUTPUT_ROOT, SHARD_DIR, ANALYSIS_DIR,
    FIGURE_DIR, COMPARISON_DIR,
):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

for required in (
    MANIFEST_PATH, MANIFEST_LOCK_PATH,
    REFERENCE_PAIRED_PATH, REFERENCE_DECISION_PATH,
):
    assert required.is_file(), f'Missing frozen prerequisite: {required}'

claim_registry = {
    'schema_version': 'visualtorque-large-model-frozen-claim-1.0',
    'analysis_status': 'confirmatory_scale_replication',
    'target_model_id': MODEL_ID,
    'target_model_revision': MODEL_REVISION,
    'reference_model_id': REFERENCE_MODEL_ID,
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'manifest_reuse': 'exact prior 48-triplet orthogonal manifest; no resampling',
    'layer_mapping_rule': (
        'round((reference_layer / (32 - 1)) * (64 - 1)); '
        'no target-model layer search'
    ),
    'reference_sites': [list(site) for site in REFERENCE_SITES],
    'target_sites': [list(site) for site in FROZEN_SITES],
    'tasks': list(TASKS),
    'donor_types': list(DONOR_TYPES),
    'primary_estimand': (
        'denominator-free shift in the changed-vs-base candidate '
        'log-probability margin'
    ),
    'primary_chain_gates': {
        'factor_binding': (
            'weight and distance factor selectivity positive at a frozen '
            'middle site after multiplicity correction'
        ),
        'side_torque_composition': (
            'both equal-torque donor effects positive and their difference '
            'equivalent within 0.25 natural-margin units'
        ),
        'late_outcome_use': (
            'both donor effects positive and equivalent at late_answer'
        ),
        'full_chain': (
            'factor_binding AND side_torque_composition AND late_outcome_use'
        ),
    },
    'predeclared_branches': [
        'scale_stable_weight_dominant_incomplete_chain',
        'scale_improves_composition_without_complete_use',
        'scale_transition_to_complete_chain',
        'scale_changes_route_but_chain_remains_incomplete',
        'large_model_factor_binding_not_confirmed',
    ],
    'equivalence_fraction': EQUIVALENCE_FRACTION,
    'cross_scale_standardized_equivalence_bound': (
        CROSS_SCALE_EQUIVALENCE_BOUND
    ),
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'sign_flip_replicates': SIGN_FLIP_REPLICATES,
    'bootstrap_unit': 'orthogonal triplet',
    'seed': SEED,
    'interpretation_limit': (
        'Grouped decoder-output transport at depth-mapped sites is not a '
        'unique circuit or proof of symbolic multiplication.'
    ),
}
PROTOCOL_LOCK_PATH = EXPERIMENT_ROOT / 'claim_registry.json'
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == claim_registry, (
        'The frozen claim registry changed. Use a new experiment version.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': claim_registry,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('Target:', MODEL_ID, '@', MODEL_REVISION)
print('Reference → target sites:', list(zip(REFERENCE_SITES, FROZEN_SITES)))
print('Manifest:', claim_registry['manifest_sha256'])
print('Output:', OUTPUT_ROOT)


In [ ]:
%%cell_progress 5/17 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


## 2. Exact frozen orthogonal support


In [ ]:
%%cell_progress 6/17 Verify the exact frozen orthogonal manifest
cell_stage('Checking manifest identity and exact Qwen-9B support')
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value}'.encode('utf-8')).hexdigest()

factorial_manifest = read_jsonl(MANIFEST_PATH)
assert len(factorial_manifest) == EXPECTED_TRIPLETS
assert len({str(row['triplet_id']) for row in factorial_manifest}) == EXPECTED_TRIPLETS
all_scene_ids = {
    str(row[key])
    for row in factorial_manifest
    for key in ('base_scene_id', 'weight_scene_id', 'distance_scene_id')
}
assert len(all_scene_ids) == 3 * EXPECTED_TRIPLETS
for row in factorial_manifest:
    assert len(set(map(int, row['scalar_values']))) == 4
    assert abs(int(row['unit_step'])) != abs(int(row['distance_step']))
    assert set(row['task_values']) == set(TASKS)
    assert (
        row['task_values']['side_torque']['weight_source']
        == row['task_values']['side_torque']['distance_source']
        != row['task_values']['side_torque']['base']
    )
    assert (
        row['task_values']['net_torque']['weight_source']
        == row['task_values']['net_torque']['distance_source']
        != row['task_values']['net_torque']['base']
    )

manifest_lock = json.loads(MANIFEST_LOCK_PATH.read_text(encoding='utf-8'))
lock_identity = manifest_lock.get('identity', manifest_lock)
locked_ids = lock_identity.get('triplet_ids')
if locked_ids is not None:
    assert list(map(str, locked_ids)) == [
        str(row['triplet_id']) for row in factorial_manifest
    ]

reference_rows = pd.read_csv(REFERENCE_PAIRED_PATH)
reference_rows['triplet_id'] = reference_rows['triplet_id'].astype(str)
assert set(reference_rows['triplet_id']) == {
    str(row['triplet_id']) for row in factorial_manifest
}
assert len(reference_rows) == EXPECTED_TRIPLETS * len(TASKS) * 4
assert not reference_rows.duplicated([
    'triplet_id', 'task', 'site_label'
]).any()
reference_decision = json.loads(
    REFERENCE_DECISION_PATH.read_text(encoding='utf-8')
)
assert reference_decision['manifest_sha256'] == sha256_file(MANIFEST_PATH)

scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(row['scene_id']): row for row in scene_records}
assert all_scene_ids <= set(scene_by_id)

example_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=EXPERIMENT_ROOT / '_unused_behavior_loader',
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'),
    batch_size=1, generation_every=0, seed=SEED,
)
outcome_examples = tuple(load_behavior_examples(example_config))
outcome_by_scene = {
    str(example.scene_id): example
    for example in outcome_examples
    if example.variant == FIXED_VARIANT
}
assert len(outcome_by_scene) == len(scene_records)

manifest_frame = pd.DataFrame([
    {key: value for key, value in row.items() if key != 'task_values'}
    for row in factorial_manifest
])
manifest_frame.to_csv(
    EXPERIMENT_ROOT / 'frozen_orthogonal_manifest_view.csv', index=False
)
print('Exact frozen triplets:', len(factorial_manifest))
print('Scene-disjoint images:', len(all_scene_ids))
print('Reference paired rows:', len(reference_rows))
display(manifest_frame.groupby([
    'changed_side', 'factor_direction', 'outcome_changed'
]).size().rename('n').reset_index())


In [ ]:
%%cell_progress 7/17 Load Qwen3.5-27B and verify depth-mapped layers
cell_stage('Loading pinned 27B weights')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT, (
    LAYER_PATH, len(DECODER_LAYERS), EXPECTED_LAYER_COUNT
)
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
device_map = getattr(adapter.model, 'hf_device_map', {})
disk_modules = [name for name, device in device_map.items() if device == 'disk']
assert not disk_modules, (
    'Disk offload is too slow for the confirmatory capture: ', disk_modules[:5]
)
print('Layer path:', LAYER_PATH)
print('Decoder layers:', len(DECODER_LAYERS))
print('Capture layers:', CAPTURE_LAYERS)
print('Device map summary:', sorted(set(map(str, device_map.values()))))
print('Allocated GPU GB:', round(torch.cuda.memory_allocated() / 2**30, 2))


## 3. Frozen representation, composition, and answer-use transport


In [ ]:
%%cell_progress 8/17 Define orthogonal tasks, diagnostic options, and exact masks
cell_stage('Defining factor, cross-value, torque, and outcome queries')
import math
from dataclasses import replace

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    "intervals from the central pivot. A stack's turning effect is its "
    'number of squares multiplied by its distance. Add turning effects '
    'within each side and compare the two side totals.'
)

def task_question(task, side):
    if task == 'side_units':
        return f'How many blocks are on the {side} side?'
    if task == 'side_distance':
        return (
            f'How many tick intervals from the pivot is the stack on the '
            f'{side} side?'
        )
    if task == 'side_torque':
        return f'What is the total {side}-side torque?'
    if task == 'net_torque':
        return 'What is left torque minus right torque?'
    if task == 'outcome':
        return 'Which side will rotate downward?'
    raise KeyError(task)

def option_semantics(row, task):
    values = row['task_values'][task]
    if task == 'outcome':
        semantics = ['left', 'balanced', 'right']
    elif task == 'side_units':
        semantics = [
            int(values['base']),
            int(values['weight_source']),
            int(row['task_values']['side_distance']['distance_source']),
        ]
    elif task == 'side_distance':
        semantics = [
            int(values['base']),
            int(values['distance_source']),
            int(row['task_values']['side_units']['weight_source']),
        ]
    else:
        semantics = []
        for name in ('base', 'weight_source', 'distance_source'):
            value = int(values[name])
            if value not in semantics:
                semantics.append(value)
        anchor = semantics[0]
        for value in (
            anchor + 1, anchor - 1, 0, 1, -1,
            anchor + 2, anchor - 2, anchor + 3, anchor - 3,
        ):
            if len(semantics) == 3:
                break
            if value not in semantics:
                semantics.append(value)
    assert len(semantics) == 3 and len(set(semantics)) == 3, (task, semantics)
    offset = int(stable_key(
        f"options:{row['triplet_id']}:{task}"
    )[:8], 16) % 3
    return tuple(semantics[offset:] + semantics[:offset])

def display_value(task, value):
    if task != 'outcome':
        return str(value)
    return {
        'left': 'left side',
        'balanced': 'balanced',
        'right': 'right side',
    }[str(value)]

def custom_prompt(task, side, semantics):
    labels = ('A', 'B', 'C')
    options = '\n'.join(
        f'{label}. {display_value(task, value)}'
        for label, value in zip(labels, semantics)
    )
    return '\n\n'.join([
        'Solve this balance-scale question using only the supplied scene.',
        RULE_TEXT,
        'Question: ' + task_question(task, side),
        'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option '
        'letter and nothing else. Valid replies are: A, B, C.',
    ])

def make_task_bundle(row, task):
    side = str(row['changed_side'])
    semantics = option_semantics(row, task)
    labels = ('A', 'B', 'C')
    prompt = custom_prompt(task, side, semantics)
    scene_ids = {
        'base': str(row['base_scene_id']),
        'weight_source': str(row['weight_scene_id']),
        'distance_source': str(row['distance_scene_id']),
    }
    examples = {}
    for name, scene_id in scene_ids.items():
        value = row['task_values'][task][name]
        value = str(value) if task == 'outcome' else int(value)
        assert value in semantics, (task, name, value, semantics)
        base = outcome_by_scene[scene_id]
        gold_index = semantics.index(value)
        examples[name] = replace(
            base,
            example_id=(
                f"image_rule:scale_{row['triplet_id']}_{task}_{name}"
            ),
            question_id=f"scale_{row['triplet_id']}_{task}",
            question_kind=task,
            variant='large_model_frozen_claim_fixed_options',
            prompt=prompt,
            labels=labels,
            option_text=tuple(
                display_value(task, value) for value in semantics
            ),
            option_semantics=semantics,
            gold_label=labels[gold_index],
            gold_semantic=value,
        )
    goal_name = {
        'side_units': 'weight_source',
        'side_distance': 'distance_source',
        'side_torque': 'weight_source',
        'net_torque': 'weight_source',
        'outcome': 'weight_source',
    }[task]
    cross_semantic = None
    if task == 'side_units':
        cross_semantic = int(
            row['task_values']['side_distance']['distance_source']
        )
    elif task == 'side_distance':
        cross_semantic = int(
            row['task_values']['side_units']['weight_source']
        )
    if cross_semantic is not None:
        assert cross_semantic in semantics
        assert cross_semantic not in {
            examples['base'].gold_semantic,
            examples[goal_name].gold_semantic,
        }
    return {
        'task': task,
        'semantics': semantics,
        'examples': examples,
        'base_semantic': examples['base'].gold_semantic,
        'goal_semantic': examples[goal_name].gold_semantic,
        'cross_semantic': cross_semantic,
        'goal_name': goal_name,
        'estimable': (
            examples['base'].gold_semantic
            != examples[goal_name].gold_semantic
        ),
    }

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(
        output.last_hidden_state
    ):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )),
        fixed_prefix,
    )
    return {
        'examples': examples,
        'inputs': inputs,
        'token_by_label': token_by_label,
    }

def discover_masks(prepared):
    assert len(prepared['examples']) == 1
    example = prepared['examples'][0]
    ids = prepared['inputs']['input_ids'][0].detach().cpu()
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    tokenizer = getattr(adapter.processor, 'tokenizer', adapter.processor)
    tokens = tokenizer.convert_ids_to_tokens(ids.tolist())
    image_ids = set()
    if IMAGE_TOKEN_ID_OVERRIDE is not None:
        values = (
            IMAGE_TOKEN_ID_OVERRIDE
            if isinstance(IMAGE_TOKEN_ID_OVERRIDE, (list, tuple))
            else [IMAGE_TOKEN_ID_OVERRIDE]
        )
        image_ids.update(int(value) for value in values)
    for token_id, token_text in zip(ids.tolist(), tokens):
        lower = str(token_text).lower()
        if any(marker in lower for marker in ('image', 'vision', '<img', 'img>')):
            image_ids.add(int(token_id))
    image = torch.tensor(
        [int(token_id) in image_ids for token_id in ids.tolist()],
        dtype=torch.bool,
    ) & attention
    assert bool(image.any()), 'No image tokens detected.'
    final = torch.zeros_like(attention)
    final[int(torch.where(attention)[0][-1])] = True
    all_text = attention & ~image & ~final

    encoding = tokenizer(
        example.prompt, add_special_tokens=False,
        return_offsets_mapping=True,
    )
    prompt_ids = list(encoding['input_ids'])
    offsets = list(encoding['offset_mapping'])
    id_list = ids.tolist()
    starts = [
        start for start in range(len(ids) - len(prompt_ids) + 1)
        if id_list[start:start + len(prompt_ids)] == prompt_ids
    ]
    assert len(starts) == 1, len(starts)
    prompt_start = starts[0]
    sections = example.prompt.split('\n\n')
    assert len(sections) == 5
    names = (
        'intro_tokens', 'rule_tokens', 'question_tokens',
        'options_tokens', 'answer_instruction_tokens',
    )
    masks = {
        'valid_tokens': attention,
        'image_tokens': image,
        'all_text_except_final': all_text,
        'final_prompt_token': final,
    }
    spans = []
    cursor = 0
    for name, section in zip(names, sections):
        start = example.prompt.find(section, cursor)
        end = start + len(section)
        cursor = end
        spans.append((name, start, end))
        masks[name] = torch.zeros_like(attention)
    for offset_index, (token_start, token_end) in enumerate(offsets):
        overlaps = [
            (max(0, min(token_end, end) - max(token_start, start)), name)
            for name, start, end in spans
        ]
        overlap, name = max(overlaps)
        full_index = prompt_start + offset_index
        if overlap > 0 and bool(all_text[full_index]):
            masks[name][full_index] = True
    covered = torch.zeros_like(attention)
    for name in names:
        covered |= masks[name]
    masks['separator_chat_tokens'] = all_text & ~covered
    assert torch.equal(
        masks['options_tokens'] | masks['intro_tokens']
        | masks['rule_tokens'] | masks['question_tokens']
        | masks['answer_instruction_tokens']
        | masks['separator_chat_tokens'],
        all_text,
    )
    masks['tokens'] = tokens
    return masks

def assert_aligned(prepared_by_name):
    names = list(prepared_by_name)
    reference = prepared_by_name[names[0]]
    reference_masks = discover_masks(reference)
    for name in names[1:]:
        current = prepared_by_name[name]
        for key in ('input_ids', 'attention_mask'):
            assert torch.equal(
                reference['inputs'][key], current['inputs'][key]
            ), (name, key)
        masks = discover_masks(current)
        for _, scope, _ in FROZEN_SITES:
            assert torch.equal(reference_masks[scope], masks[scope]), (
                name, scope
            )
    return reference_masks

print('Frozen scale-replication prompts and exact masks ready.')


In [ ]:
%%cell_progress 9/17 Define decoder capture and batched patch scorers
cell_stage('Defining candidate-space scoring and decoder patching')
from collections import defaultdict

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for row_index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[
                row_index, prepared['token_by_label'][label]
            ].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        total = sum(weights)
        probabilities = [value / total for value in weights]
        predicted_index = int(np.argmax(scores))
        records.append({
            'scene_id': str(example.scene_id),
            'example_id': str(example.example_id),
            'task': str(example.question_kind),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_semantic': example.option_semantics[predicted_index],
            'gold_semantic': example.gold_semantic,
            'correct': (
                example.option_semantics[predicted_index]
                == example.gold_semantic
            ),
        })
    return records

def forward_capture_batch(prepared):
    hidden_by_layer = {}
    handles = []
    for layer_index in CAPTURE_LAYERS:
        def hook(_module, _inputs, output, layer_index=layer_index):
            hidden_by_layer[layer_index] = (
                output_hidden_tensor(output).detach().cpu().clone()
            )
        handles.append(
            DECODER_LAYERS[layer_index].register_forward_hook(hook)
        )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits), hidden_by_layer

def forward_patch_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layer_indices = {int(row['layer_index']) for row in conditions}
    assert len(layer_indices) == 1
    layer_index = next(iter(layer_indices))
    donors = torch.stack([row['donor_hidden'] for row in conditions])
    masks = torch.stack([row['patch_mask'] for row in conditions])
    audit = {}

    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        donor = donors.to(device=hidden.device, dtype=hidden.dtype)
        mask = masks.to(device=hidden.device)
        assert hidden.shape == donor.shape
        modified = hidden.clone()
        modified[mask] = donor[mask]
        audit['counts'] = mask.sum(dim=1).detach().cpu().tolist()
        return replace_output_hidden(output, modified)

    handle = DECODER_LAYERS[layer_index].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, count in zip(records, conditions, audit['counts']):
        record.update({
            key: value for key, value in condition.items()
            if key not in ('donor_hidden', 'patch_mask')
        })
        record['patched_tokens'] = int(count)
    return records

def score_conditions(target_example, prepared_cache, conditions):
    records = []
    grouped = defaultdict(list)
    for condition in conditions:
        grouped[int(condition['layer_index'])].append(condition)

    def run_chunk(chunk):
        actual_size = len(chunk)
        assert 1 <= actual_size <= PATCH_CONDITION_BATCH_SIZE
        padded = list(chunk)
        while len(padded) < PATCH_CONDITION_BATCH_SIZE:
            padded.append(padded[-1])
        size = len(padded)
        prepared = prepared_cache.get(size)
        if prepared is None:
            prepared = prepare_examples([target_example] * size)
            prepared_cache[size] = prepared
        try:
            return forward_patch_chunk(prepared, padded)[:actual_size]
        except torch.cuda.OutOfMemoryError as exc:
            prepared_cache.pop(size, None)
            torch.cuda.empty_cache()
            raise RuntimeError(
                'Fixed batch-size patching ran out of memory. Restart with '
                'an A100; changing batch size changes numerical parity.'
            ) from exc

    for group in grouped.values():
        for start in range(0, len(group), PATCH_CONDITION_BATCH_SIZE):
            records.extend(run_chunk(
                group[start:start + PATCH_CONDITION_BATCH_SIZE]
            ))
    return records

print('Capture and patch scorers ready.')


In [ ]:
%%cell_progress 10/17 Run mandatory frozen-support, alignment, and parity smoke tests
cell_stage('Checking physical equalities, value separation, masks, and parity')
smoke_row = factorial_manifest[0]
assert len(set(smoke_row['scalar_values'])) == 4
assert abs(int(smoke_row['unit_step'])) != abs(int(smoke_row['distance_step']))
assert (
    smoke_row['task_values']['side_units']['weight_source']
    != smoke_row['task_values']['side_units']['base']
)
assert (
    smoke_row['task_values']['side_units']['distance_source']
    == smoke_row['task_values']['side_units']['base']
)
assert (
    smoke_row['task_values']['side_distance']['distance_source']
    != smoke_row['task_values']['side_distance']['base']
)
assert (
    smoke_row['task_values']['side_torque']['weight_source']
    == smoke_row['task_values']['side_torque']['distance_source']
)
assert (
    smoke_row['task_values']['net_torque']['weight_source']
    == smoke_row['task_values']['net_torque']['distance_source']
)

factor_bundle = make_task_bundle(smoke_row, 'side_units')
assert factor_bundle['cross_semantic'] in factor_bundle['semantics']
smoke_bundle = make_task_bundle(smoke_row, 'net_torque')
smoke_prepared = {
    name: prepare_examples([example])
    for name, example in smoke_bundle['examples'].items()
}
smoke_masks = assert_aligned(smoke_prepared)
for _, scope, _ in FROZEN_SITES:
    assert bool(smoke_masks[scope].any()), scope

smoke_target_batch = prepare_examples([
    smoke_bundle['examples']['base']
] * PATCH_CONDITION_BATCH_SIZE)
smoke_source_batch = prepare_examples([
    smoke_bundle['examples']['weight_source']
] * PATCH_CONDITION_BATCH_SIZE)
smoke_target_scores, smoke_target_hidden = forward_capture_batch(smoke_target_batch)
smoke_source_scores, smoke_source_hidden = forward_capture_batch(smoke_source_batch)
smoke_layer, smoke_scope = SITE_BY_LABEL['middle_text']
smoke_conditions = [
    {
        'condition': 'self_all_tokens', 'task': 'net_torque',
        'donor_type': 'self_base', 'site_label': 'smoke_all_tokens',
        'scope': 'valid_tokens', 'layer_index': smoke_layer,
        'donor_scene_id': smoke_row['base_scene_id'],
        'donor_hidden': smoke_target_hidden[smoke_layer][0],
        'patch_mask': smoke_masks['valid_tokens'],
    },
    {
        'condition': 'source_all_tokens', 'task': 'net_torque',
        'donor_type': 'weight_source', 'site_label': 'smoke_all_tokens',
        'scope': 'valid_tokens', 'layer_index': smoke_layer,
        'donor_scene_id': smoke_row['weight_scene_id'],
        'donor_hidden': smoke_source_hidden[smoke_layer][0],
        'patch_mask': smoke_masks['valid_tokens'],
    },
]
smoke_records = score_conditions(
    smoke_bundle['examples']['base'], {}, smoke_conditions
)

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

self_error = float(np.max(np.abs(
    centered_scores(smoke_records[0]) - centered_scores(smoke_target_scores[0])
)))
source_error = float(np.max(np.abs(
    centered_scores(smoke_records[1]) - centered_scores(smoke_source_scores[0])
)))
print('Centered self/source parity errors:', self_error, source_error)
assert self_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, self_error
assert source_error < SMOKE_CENTERED_LOGPROB_TOLERANCE, source_error
print('Smoke triplet:', smoke_row['triplet_id'])
print('Scalar values:', smoke_row['scalar_values'])
print('Frozen-site token counts:', {
    f'{label}:L{layer}:{scope}': int(smoke_masks[scope].sum())
    for layer, scope, label in FROZEN_SITES
})
del (
    smoke_prepared, smoke_target_batch, smoke_source_batch,
    smoke_target_hidden, smoke_source_hidden,
)
torch.cuda.empty_cache()


In [ ]:
%%cell_progress 11/17 Capture or resume frozen 27B interventions
cell_stage('Freezing capture identity and checking completed shards')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

CONDITIONS_PER_TASK = len(FROZEN_SITES) * len(DONOR_TYPES)
RECORDS_PER_TRIPLET = len(TASKS) * CONDITIONS_PER_TASK
EXPECTED_PATCH_RECORDS = len(factorial_manifest) * RECORDS_PER_TRIPLET
capture_identity = {
    'schema_version': 'large-model-frozen-claim-capture-1.0',
    'model_key': MODEL_KEY,
    'model_id': MODEL_ID,
    'resolved_revision': MODEL_REVISION,
    'dtype': MODEL_DTYPE,
    'dataset_fingerprint': example_config.dataset_fingerprint(),
    'protocol_lock_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'frozen_sites': [list(site) for site in FROZEN_SITES],
    'tasks': list(TASKS),
    'donor_types': list(DONOR_TYPES),
    'expected_triplets': len(factorial_manifest),
    'records_per_triplet': RECORDS_PER_TRIPLET,
    'expected_patch_records': EXPECTED_PATCH_RECORDS,
    'fixed_patch_batch_size': PATCH_CONDITION_BATCH_SIZE,
    'partial_batches_are_padded': True,
    'layer_path': LAYER_PATH,
    'seed': SEED,
    'reference_model_id': REFERENCE_MODEL_ID,
    'reference_sites': [list(site) for site in REFERENCE_SITES],
    'depth_mapping_rule': claim_registry['layer_mapping_rule'],
}
capture_config_path = OUTPUT_ROOT / 'capture_config.json'
if capture_config_path.is_file():
    previous = json.loads(capture_config_path.read_text(encoding='utf-8'))
    assert previous.get('identity') == capture_identity, (
        'Existing model capture differs. Use a new output version.'
    )
else:
    write_json_atomic(capture_config_path, {
        'identity': capture_identity,
        'adapter_metadata': adapter.metadata,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    })
CAPTURE_SHA256 = sha256_file(capture_config_path)

completed_triplet_ids = set()
completed_records = 0
for path in sorted(SHARD_DIR.glob('shard_triplet_*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    triplet_id = str(payload['triplet']['triplet_id'])
    assert triplet_id not in completed_triplet_ids
    assert len(payload['patch_records']) == RECORDS_PER_TRIPLET
    completed_triplet_ids.add(triplet_id)
    completed_records += len(payload['patch_records'])

progress = tqdm(
    total=EXPECTED_PATCH_RECORDS, initial=completed_records,
    desc=f'{MODEL_KEY} frozen scale patches', unit='condition',
    dynamic_ncols=True,
)
for triplet_index, row in enumerate(factorial_manifest):
    triplet_id = str(row['triplet_id'])
    if triplet_id in completed_triplet_ids:
        continue
    progress.set_postfix_str(
        f"{row['changed_side']} | dTau={row['delta_net_torque']}"
    )
    triplet_baselines = {}
    triplet_patch_records = []
    for task in TASKS:
        bundle = make_task_bundle(row, task)
        examples = bundle['examples']
        prepared = {
            name: prepare_examples([example])
            for name, example in examples.items()
        }
        masks = assert_aligned(prepared)
        capture_names = ('base', 'weight_source', 'distance_source')
        scores = {}
        hidden = {}
        for capture_name in capture_names:
            capture_prepared = prepare_examples([examples[capture_name]])
            capture_scores, capture_hidden = forward_capture_batch(
                capture_prepared
            )
            scores[capture_name] = capture_scores[0]
            hidden[capture_name] = {
                layer_index: value[0]
                for layer_index, value in capture_hidden.items()
            }
            del capture_prepared, capture_scores, capture_hidden
            torch.cuda.empty_cache()

        triplet_baselines[task] = {
            'task': task,
            'semantics': list(bundle['semantics']),
            'base_semantic': bundle['base_semantic'],
            'goal_semantic': bundle['goal_semantic'],
            'cross_semantic': bundle['cross_semantic'],
            'goal_name': bundle['goal_name'],
            'estimable': bool(bundle['estimable']),
            'scores': scores,
        }

        conditions = []
        for layer_index, scope, site_label in FROZEN_SITES:
            for donor_type in DONOR_TYPES:
                conditions.append({
                    'analysis_family': 'large_model_frozen_claim',
                    'condition': (
                        f'{task}__{donor_type}__{site_label}'
                    ),
                    'task': task,
                    'donor_type': donor_type,
                    'site_label': site_label,
                    'scope': scope,
                    'layer_index': int(layer_index),
                    'donor_scene_id': str(
                        row[f'{donor_type.split("_")[0]}_scene_id']
                    ),
                    'donor_hidden': hidden[donor_type][layer_index],
                    'patch_mask': masks[scope],
                })
        prepared_cache = {1: prepared['base']}
        records = score_conditions(
            examples['base'], prepared_cache, conditions
        )
        triplet_patch_records.extend(records)
        progress.update(len(records))
        del prepared, hidden, prepared_cache
        torch.cuda.empty_cache()

    assert len(triplet_patch_records) == RECORDS_PER_TRIPLET
    payload = {
        'schema_version': 'large-model-frozen-claim-triplet-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'triplet': row,
        'baselines': triplet_baselines,
        'patch_records': triplet_patch_records,
    }
    shard_path = SHARD_DIR / f'shard_triplet_{triplet_index:04d}.json'
    write_json_atomic(shard_path, payload)
progress.close()

completed = sorted(SHARD_DIR.glob('shard_triplet_*.json'))
assert len(completed) == len(factorial_manifest)
observed_records = 0
for path in completed:
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    assert len(payload['patch_records']) == RECORDS_PER_TRIPLET
    observed_records += len(payload['patch_records'])
assert observed_records == EXPECTED_PATCH_RECORDS
write_json_atomic(OUTPUT_ROOT / 'capture_complete.json', {
    'triplets': len(completed),
    'patch_records': observed_records,
    'capture_identity_sha256': CAPTURE_SHA256,
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'peak_gpu_memory_gb': round(torch.cuda.max_memory_allocated() / 2**30, 3),
})
print('Capture complete:', observed_records, 'patch records')


## 4. Frozen inference and exact scale synthesis


In [ ]:
%%cell_progress 12/17 Release GPU and assemble large-model metrics
cell_stage('Releasing model memory')
del adapter
gc.collect()
torch.cuda.empty_cache()

cell_stage('Reading triplet shards and computing pair-level effects')
long_rows = []
paired_rows = []

def centered_vector(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def semantic_margin(record, goal_semantic, base_semantic):
    semantics = list(record['candidate_semantics'])
    goal_index = semantics.index(goal_semantic)
    base_index = semantics.index(base_semantic)
    scores = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(scores[goal_index] - scores[base_index])

def cosine(left, right):
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    denominator = float(np.linalg.norm(left) * np.linalg.norm(right))
    if denominator <= 1e-12:
        return np.nan
    return float(np.dot(left, right) / denominator)

for path in tqdm(
    sorted(SHARD_DIR.glob('shard_triplet_*.json')),
    desc='Assembling frozen scale metrics', unit='triplet',
):
    payload = json.loads(path.read_text(encoding='utf-8'))
    row = payload['triplet']
    patches = pd.DataFrame(payload['patch_records'])
    for task in TASKS:
        baseline = payload['baselines'][task]
        base_record = baseline['scores']['base']
        weight_record = baseline['scores']['weight_source']
        distance_record = baseline['scores']['distance_source']
        base_semantic = baseline['base_semantic']
        goal_semantic = baseline['goal_semantic']
        cross_semantic = baseline.get('cross_semantic')
        estimable = bool(baseline['estimable'])
        base_vector = centered_vector(base_record)
        goal_record = baseline['scores'][baseline['goal_name']]
        goal_vector = centered_vector(goal_record)
        natural_vector = goal_vector - base_vector
        if estimable:
            base_margin = semantic_margin(base_record, goal_semantic, base_semantic)
            goal_margin = semantic_margin(goal_record, goal_semantic, base_semantic)
            natural_margin = goal_margin - base_margin
        else:
            base_margin = goal_margin = natural_margin = np.nan
        base_cross_margin = (
            semantic_margin(base_record, cross_semantic, base_semantic)
            if cross_semantic is not None else np.nan
        )

        task_rows = []
        task_patches = patches[patches['task'] == task]
        for _, patch in task_patches.iterrows():
            patch_record = patch.to_dict()
            patch_vector = centered_vector(patch_record)
            displacement = patch_vector - base_vector
            if estimable:
                patch_margin = semantic_margin(
                    patch_record, goal_semantic, base_semantic
                )
                margin_shift = patch_margin - base_margin
                recovery = (
                    margin_shift / natural_margin
                    if abs(natural_margin) >= NATURAL_EFFECT_EPS else np.nan
                )
            else:
                patch_margin = margin_shift = recovery = np.nan
            cross_value_shift = (
                semantic_margin(patch_record, cross_semantic, base_semantic)
                - base_cross_margin
                if cross_semantic is not None else np.nan
            )
            record = {
                'model_key': MODEL_KEY,
                'model_id': MODEL_ID,
                'triplet_id': row['triplet_id'],
                'changed_side': row['changed_side'],
                'factor_direction': row['factor_direction'],
                'delta_side_torque': row['delta_side_torque'],
                'delta_net_torque': row['delta_net_torque'],
                'outcome_changed': bool(row['outcome_changed']),
                'any_prior_scene_overlap': bool(row['any_prior_scene_overlap']),
                'task': task,
                'estimable': estimable,
                'donor_type': str(patch['donor_type']),
                'site_label': str(patch['site_label']),
                'layer_index': int(patch['layer_index']),
                'scope': str(patch['scope']),
                'base_semantic': base_semantic,
                'goal_semantic': goal_semantic,
                'cross_semantic': cross_semantic,
                'margin_shift': margin_shift,
                'cross_value_shift': cross_value_shift,
                'natural_margin': natural_margin,
                'recovery': recovery,
                'vector_alignment_to_natural': cosine(displacement, natural_vector),
                'displacement_0': float(displacement[0]),
                'displacement_1': float(displacement[1]),
                'displacement_2': float(displacement[2]),
                'base_correct': bool(base_record['correct']),
                'weight_source_correct': bool(weight_record['correct']),
                'distance_source_correct': bool(distance_record['correct']),
                'all_three_correct': bool(
                    base_record['correct'] and weight_record['correct']
                    and distance_record['correct']
                ),
                'patched_tokens': int(patch['patched_tokens']),
            }
            long_rows.append(record)
            task_rows.append(record)

        for _, _, site_label in FROZEN_SITES:
            site_records = [
                value for value in task_rows if value['site_label'] == site_label
            ]
            assert len(site_records) == 2
            by_donor = {value['donor_type']: value for value in site_records}
            weight_effect = by_donor['weight_source']['margin_shift']
            distance_effect = by_donor['distance_source']['margin_shift']
            weight_disp = np.asarray([
                by_donor['weight_source'][f'displacement_{index}']
                for index in range(3)
            ])
            distance_disp = np.asarray([
                by_donor['distance_source'][f'displacement_{index}']
                for index in range(3)
            ])
            if task == 'side_units':
                relevant_effect = weight_effect
                irrelevant_effect = distance_effect
                factor_selectivity = relevant_effect - irrelevant_effect
                cross_value_leakage = by_donor['distance_source']['cross_value_shift']
                cross_source_record = distance_record
            elif task == 'side_distance':
                relevant_effect = distance_effect
                irrelevant_effect = weight_effect
                factor_selectivity = relevant_effect - irrelevant_effect
                cross_value_leakage = by_donor['weight_source']['cross_value_shift']
                cross_source_record = weight_record
            else:
                relevant_effect = irrelevant_effect = np.nan
                factor_selectivity = cross_value_leakage = np.nan
                cross_source_record = None
            cross_value_behavior_error = (
                bool(cross_source_record['predicted_semantic'] == cross_semantic)
                if cross_source_record is not None else False
            )
            paired_rows.append({
                'model_key': MODEL_KEY,
                'model_id': MODEL_ID,
                'triplet_id': row['triplet_id'],
                'changed_side': row['changed_side'],
                'factor_direction': row['factor_direction'],
                'delta_side_torque': row['delta_side_torque'],
                'delta_net_torque': row['delta_net_torque'],
                'outcome_changed': bool(row['outcome_changed']),
                'any_prior_scene_overlap': bool(row['any_prior_scene_overlap']),
                'task': task,
                'estimable': estimable,
                'site_label': site_label,
                'weight_effect': weight_effect,
                'distance_effect': distance_effect,
                'relevant_route_effect': relevant_effect,
                'irrelevant_route_effect': irrelevant_effect,
                'factor_selectivity': factor_selectivity,
                'cross_value_leakage': cross_value_leakage,
                'cross_value_behavior_error': cross_value_behavior_error,
                'mean_donor_effect': (
                    np.nanmean([weight_effect, distance_effect])
                    if estimable else np.nan
                ),
                'donor_effect_difference': (
                    weight_effect - distance_effect if estimable else np.nan
                ),
                'abs_donor_gap': (
                    abs(weight_effect - distance_effect) if estimable else np.nan
                ),
                'displacement_cosine': cosine(weight_disp, distance_disp),
                'natural_margin': natural_margin,
                'base_correct': bool(base_record['correct']),
                'weight_source_correct': bool(weight_record['correct']),
                'distance_source_correct': bool(distance_record['correct']),
                'all_three_correct': by_donor['weight_source']['all_three_correct'],
            })

long_frame = pd.DataFrame(long_rows)
paired_frame = pd.DataFrame(paired_rows)
expected_long = (
    len(factorial_manifest) * len(TASKS)
    * len(FROZEN_SITES) * len(DONOR_TYPES)
)
expected_paired = len(factorial_manifest) * len(TASKS) * len(FROZEN_SITES)
assert len(long_frame) == expected_long
assert len(paired_frame) == expected_paired
assert not long_frame.duplicated([
    'triplet_id', 'task', 'site_label', 'donor_type'
]).any()
long_frame.to_csv(ANALYSIS_DIR / 'patch_level_metrics.csv', index=False)
paired_frame.to_csv(ANALYSIS_DIR / 'paired_triplet_metrics.csv', index=False)
baseline_accuracy = (
    long_frame.groupby(['task'])[
        ['base_correct', 'weight_source_correct', 'distance_source_correct']
    ].mean().reset_index()
)
baseline_accuracy.to_csv(ANALYSIS_DIR / 'baseline_task_accuracy.csv', index=False)
display(baseline_accuracy)
print('Patch rows:', len(long_frame), '| Paired rows:', len(paired_frame))


In [ ]:
%%cell_progress 13/17 Run frozen representation, composition, and use tests
cell_stage('Defining deterministic paired bootstrap and sign-flip tests')

def seed_for(*values):
    digest = hashlib.sha256(
        (str(SEED) + '|' + '|'.join(map(str, values))).encode('utf-8')
    ).hexdigest()
    return int(digest[:8], 16)

def paired_summary(frame, column, family, task, site_label, population):
    values = pd.to_numeric(frame[column], errors='coerce').to_numpy(float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {
            'family': family, 'metric': column, 'task': task,
            'site_label': site_label, 'population': population,
            'n': 0, 'mean': np.nan, 'ci_low': np.nan,
            'ci_high': np.nan, 'sign_flip_p': np.nan,
        }
    rng = np.random.default_rng(seed_for(
        MODEL_KEY, family, task, site_label, population, column
    ))
    indices = rng.integers(
        0, len(values), size=(BOOTSTRAP_REPLICATES, len(values))
    )
    boot = values[indices].mean(axis=1)
    observed = float(values.mean())
    signs = rng.choice(
        (-1.0, 1.0), size=(SIGN_FLIP_REPLICATES, len(values))
    )
    null = (signs * values).mean(axis=1)
    p_value = float(
        (1 + np.sum(np.abs(null) >= abs(observed)))
        / (SIGN_FLIP_REPLICATES + 1)
    )
    return {
        'family': family, 'metric': column, 'task': task,
        'site_label': site_label, 'population': population,
        'n': int(len(values)), 'mean': observed,
        'ci_low': float(np.quantile(boot, ALPHA / 2)),
        'ci_high': float(np.quantile(boot, 1 - ALPHA / 2)),
        'sign_flip_p': p_value,
    }

metric_plan = {
    'side_units': (
        'factor_selectivity', 'relevant_route_effect',
        'irrelevant_route_effect', 'cross_value_leakage',
    ),
    'side_distance': (
        'factor_selectivity', 'relevant_route_effect',
        'irrelevant_route_effect', 'cross_value_leakage',
    ),
    'side_torque': (
        'weight_effect', 'distance_effect', 'mean_donor_effect',
        'donor_effect_difference', 'displacement_cosine',
    ),
    'net_torque': (
        'weight_effect', 'distance_effect', 'mean_donor_effect',
        'donor_effect_difference', 'displacement_cosine',
    ),
    'outcome': (
        'weight_effect', 'distance_effect', 'mean_donor_effect',
        'donor_effect_difference', 'displacement_cosine',
    ),
}
summary_rows = []
populations = {
    'all': paired_frame,
    'all_three_correct_secondary': paired_frame[paired_frame['all_three_correct']],
}
for population, population_frame in populations.items():
    for task, metrics in metric_plan.items():
        task_frame = population_frame[
            (population_frame['task'] == task) & population_frame['estimable']
        ]
        for _, _, site_label in FROZEN_SITES:
            site_frame = task_frame[task_frame['site_label'] == site_label]
            for metric in metrics:
                summary_rows.append(paired_summary(
                    site_frame, metric,
                    'factor_binding' if task in ('side_units', 'side_distance')
                    else 'torque_route' if task in ('side_torque', 'net_torque')
                    else 'outcome_use',
                    task, site_label, population,
                ))
summary_frame = pd.DataFrame(summary_rows)

def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    result = np.full(len(values), np.nan)
    valid = np.where(np.isfinite(values))[0]
    if len(valid) == 0:
        return result
    order = valid[np.argsort(values[valid])]
    ranked = values[order] * len(order) / np.arange(1, len(order) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    result[order] = np.minimum(ranked, 1.0)
    return result

summary_frame['q_value'] = np.nan
for _, indices in summary_frame.groupby([
    'population', 'family', 'metric'
]).groups.items():
    summary_frame.loc[indices, 'q_value'] = bh_adjust(
        summary_frame.loc[indices, 'sign_flip_p'].to_numpy(float)
    )

summary_frame['equivalence_bound'] = np.nan
summary_frame['equivalent'] = False
all_population = paired_frame[paired_frame['estimable']]
for task in ('side_torque', 'net_torque', 'outcome'):
    for _, _, site_label in FROZEN_SITES:
        subset = all_population[
            (all_population['task'] == task)
            & (all_population['site_label'] == site_label)
        ]
        scale_values = np.abs(pd.to_numeric(
            subset['natural_margin'], errors='coerce'
        ))
        scale_values = scale_values[np.isfinite(scale_values)]
        bound = (
            EQUIVALENCE_FRACTION * float(np.median(scale_values))
            if len(scale_values) else np.nan
        )
        mask = (
            (summary_frame['population'] == 'all')
            & (summary_frame['task'] == task)
            & (summary_frame['site_label'] == site_label)
            & (summary_frame['metric'] == 'donor_effect_difference')
        )
        summary_frame.loc[mask, 'equivalence_bound'] = bound
        summary_frame.loc[mask, 'equivalent'] = (
            (summary_frame.loc[mask, 'ci_low'] > -bound)
            & (summary_frame.loc[mask, 'ci_high'] < bound)
        )

summary_frame.to_csv(ANALYSIS_DIR / 'confirmatory_summary.csv', index=False)
display(summary_frame[
    (summary_frame['population'] == 'all')
    & summary_frame['site_label'].isin(('middle_text', 'middle_options'))
].sort_values(['task', 'site_label', 'metric']))


In [ ]:
%%cell_progress 14/17 Audit competence, physical strata, and equivalence sensitivity
cell_stage('Auditing baseline competence and correctness sensitivity')
competence_base = paired_frame[[
    'triplet_id', 'task', 'base_correct', 'weight_source_correct',
    'distance_source_correct', 'all_three_correct',
]].drop_duplicates()
competence_counts = competence_base.groupby('task').agg(
    triplets=('triplet_id', 'nunique'),
    base_accuracy=('base_correct', 'mean'),
    weight_source_accuracy=('weight_source_correct', 'mean'),
    distance_source_accuracy=('distance_source_correct', 'mean'),
    all_three_correct_rate=('all_three_correct', 'mean'),
).reset_index()
competence_counts.to_csv(ANALYSIS_DIR / 'competence_counts.csv', index=False)

correctness_rows = []
for site_label in ('middle_text', 'middle_options'):
    task_frame = paired_frame[
        (paired_frame['task'] == 'side_torque')
        & (paired_frame['site_label'] == site_label)
        & paired_frame['estimable']
    ].copy()
    populations = {
        'all': np.ones(len(task_frame), dtype=bool),
        'base_correct': task_frame['base_correct'].to_numpy(bool),
        'either_source_correct': (
            task_frame['weight_source_correct']
            | task_frame['distance_source_correct']
        ).to_numpy(bool),
        'both_sources_correct': (
            task_frame['weight_source_correct']
            & task_frame['distance_source_correct']
        ).to_numpy(bool),
    }
    for population, selector in populations.items():
        subset = task_frame.loc[selector]
        correctness_rows.append(paired_summary(
            subset, 'donor_effect_difference', 'correctness_sensitivity',
            'side_torque', site_label, population,
        ))
correctness_sensitivity = pd.DataFrame(correctness_rows)
correctness_sensitivity.to_csv(
    ANALYSIS_DIR / 'correctness_sensitivity.csv', index=False
)

cell_stage('Auditing predeclared physical strata')
manifest_strata = manifest_frame[[
    'triplet_id', 'changed_side', 'factor_direction',
    'outcome_changed', 'delta_side_torque',
]].copy()
manifest_strata['triplet_id'] = manifest_strata['triplet_id'].astype(str)
manifest_strata['delta_magnitude_group'] = np.where(
    manifest_strata['delta_side_torque'].abs() <= 4,
    'small_3_4', 'large_6_12',
)
stratified = paired_frame.merge(
    manifest_strata[[
        'triplet_id', 'delta_magnitude_group'
    ]], on='triplet_id', how='left', validate='many_to_one',
)
stratum_rows = []
for site_label in ('middle_text', 'middle_options'):
    base = stratified[
        (stratified['task'] == 'side_torque')
        & (stratified['site_label'] == site_label)
        & stratified['estimable']
    ]
    for stratum in (
        'changed_side', 'factor_direction',
        'outcome_changed', 'delta_magnitude_group',
    ):
        for level, subset in base.groupby(stratum, dropna=False):
            result = paired_summary(
                subset, 'donor_effect_difference', 'physical_strata',
                'side_torque', site_label, f'{stratum}={level}',
            )
            result.update({'stratum': stratum, 'level': str(level)})
            stratum_rows.append(result)
physical_strata = pd.DataFrame(stratum_rows)
physical_strata.to_csv(ANALYSIS_DIR / 'physical_strata.csv', index=False)

cell_stage('Applying the frozen equivalence sensitivity grid')
equivalence_rows = []
for task, sites in (
    ('side_torque', ('middle_text', 'middle_options')),
    ('net_torque', ('middle_text', 'middle_options')),
    ('outcome', ('late_answer',)),
):
    for site_label in sites:
        subset = paired_frame[
            (paired_frame['task'] == task)
            & (paired_frame['site_label'] == site_label)
            & paired_frame['estimable']
        ]
        difference = paired_summary(
            subset, 'donor_effect_difference', 'equivalence_sensitivity',
            task, site_label, 'all',
        )
        natural_scale = float(np.median(np.abs(
            pd.to_numeric(subset['natural_margin'], errors='coerce').dropna()
        )))
        for fraction in EQUIVALENCE_FRACTIONS:
            bound = fraction * natural_scale
            equivalence_rows.append({
                'task': task, 'site_label': site_label,
                'fraction': fraction, 'natural_scale': natural_scale,
                'bound': bound, 'n': difference['n'],
                'difference_mean': difference['mean'],
                'ci_low': difference['ci_low'],
                'ci_high': difference['ci_high'],
                'equivalent': bool(
                    difference['ci_low'] > -bound
                    and difference['ci_high'] < bound
                ),
            })
equivalence_sensitivity = pd.DataFrame(equivalence_rows)
equivalence_sensitivity.to_csv(
    ANALYSIS_DIR / 'equivalence_sensitivity.csv', index=False
)

display(competence_counts)
display(correctness_sensitivity)
display(equivalence_sensitivity)
print('Physical-stratum rows:', len(physical_strata))


In [ ]:
%%cell_progress 15/17 Save large-model confirmatory figures
cell_stage('Rendering representation, composition, use, and competence panels')
primary = summary_frame[summary_frame['population'] == 'all']
focal_sites = ['middle_text', 'middle_options']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

ax = axes[0, 0]
for task, offset, color in (
    ('side_units', -0.06, '#2563eb'),
    ('side_distance', 0.06, '#f97316'),
):
    rows = primary[
        (primary['task'] == task)
        & (primary['metric'] == 'factor_selectivity')
    ].set_index('site_label').reindex(focal_sites)
    x = np.arange(len(focal_sites)) + offset
    y = rows['mean'].to_numpy(float)
    ax.errorbar(x, y, yerr=np.vstack([
        y - rows['ci_low'].to_numpy(float),
        rows['ci_high'].to_numpy(float) - y,
    ]), marker='o', capsize=4, label=task.replace('_', ' '), color=color)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(focal_sites)), focal_sites)
ax.set_ylabel('relevant − irrelevant donor effect')
ax.set_title('A. Factor-specific binding')
ax.legend(frameon=False)

ax = axes[0, 1]
for metric, offset, label, color in (
    ('weight_effect', -0.06, 'weight donor', '#7c3aed'),
    ('distance_effect', 0.06, 'distance donor', '#16a34a'),
):
    rows = primary[
        (primary['task'] == 'side_torque')
        & (primary['metric'] == metric)
    ].set_index('site_label').reindex(focal_sites)
    x = np.arange(len(focal_sites)) + offset
    y = rows['mean'].to_numpy(float)
    ax.errorbar(x, y, yerr=np.vstack([
        y - rows['ci_low'].to_numpy(float),
        rows['ci_high'].to_numpy(float) - y,
    ]), marker='o', capsize=4, label=label, color=color)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(focal_sites)), focal_sites)
ax.set_ylabel('changed-torque margin shift')
ax.set_title('B. Equal-torque composition routes')
ax.legend(frameon=False)

ax = axes[1, 0]
outcome_rows = primary[
    (primary['task'] == 'outcome')
    & (primary['site_label'] == 'late_answer')
    & primary['metric'].isin(('weight_effect', 'distance_effect'))
]
labels = outcome_rows['metric'].str.replace('_effect', '', regex=False)
y = outcome_rows['mean'].to_numpy(float)
ax.errorbar(np.arange(len(y)), y, yerr=np.vstack([
    y - outcome_rows['ci_low'].to_numpy(float),
    outcome_rows['ci_high'].to_numpy(float) - y,
]), fmt='o', capsize=5, color='#dc2626')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(range(len(y)), labels)
ax.set_ylabel('outcome margin shift')
ax.set_title('C. Late answer use')

ax = axes[1, 1]
accuracy = competence_counts.set_index('task').reindex(TASKS)
x = np.arange(len(TASKS))
width = 0.25
for index, (column, label) in enumerate((
    ('base_accuracy', 'base'),
    ('weight_source_accuracy', 'weight source'),
    ('distance_source_accuracy', 'distance source'),
)):
    ax.bar(x + (index - 1) * width, accuracy[column], width=width, label=label)
ax.set_xticks(x, [task.replace('_', ' ') for task in TASKS], rotation=25)
ax.set_ylim(0, 1)
ax.set_ylabel('accuracy')
ax.set_title('D. Unfiltered competence')
ax.legend(frameon=False, fontsize=8)

fig.suptitle('VisualTorque frozen-claim scale replication — Qwen3.5-27B')
fig.tight_layout()
figure_path = FIGURE_DIR / 'large_model_frozen_claim_summary.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


In [ ]:
%%cell_progress 16/17 Apply frozen chain gates
cell_stage('Evaluating representation, composition, and answer-use gates')

def summary_record(task, site_label, metric):
    rows = summary_frame[
        (summary_frame['population'] == 'all')
        & (summary_frame['task'] == task)
        & (summary_frame['site_label'] == site_label)
        & (summary_frame['metric'] == metric)
    ]
    assert len(rows) == 1, (task, site_label, metric, len(rows))
    return rows.iloc[0]

focal_sites = ('middle_text', 'middle_options')
factor_site_gates = {}
weight_dominance_site_gates = {}
side_torque_equivalence_site_gates = {}
leakage_site_flags = {}
for site_label in focal_sites:
    units = summary_record('side_units', site_label, 'factor_selectivity')
    distance = summary_record('side_distance', site_label, 'factor_selectivity')
    factor_site_gates[site_label] = bool(
        units['ci_low'] > 0 and units['q_value'] < ALPHA
        and distance['ci_low'] > 0 and distance['q_value'] < ALPHA
    )
    leakage_site_flags[site_label] = {
        task: bool(
            summary_record(task, site_label, 'cross_value_leakage')['ci_low'] > 0
            and summary_record(task, site_label, 'cross_value_leakage')['q_value'] < ALPHA
        )
        for task in ('side_units', 'side_distance')
    }
    difference = summary_record(
        'side_torque', site_label, 'donor_effect_difference'
    )
    weight_dominance_site_gates[site_label] = bool(
        difference['ci_low'] > 0 and difference['q_value'] < ALPHA
    )
    weight = summary_record('side_torque', site_label, 'weight_effect')
    distance_effect = summary_record(
        'side_torque', site_label, 'distance_effect'
    )
    side_torque_equivalence_site_gates[site_label] = bool(
        weight['ci_low'] > 0 and weight['q_value'] < ALPHA
        and distance_effect['ci_low'] > 0
        and distance_effect['q_value'] < ALPHA
        and bool(difference['equivalent'])
    )

factor_binding_supported = bool(any(factor_site_gates.values()))
weight_dominant_side_torque = bool(all(weight_dominance_site_gates.values()))
side_torque_equivalence = bool(any(side_torque_equivalence_site_gates.values()))
cross_value_leakage_detected = bool(any(
    value for site in leakage_site_flags.values() for value in site.values()
))

outcome_weight = summary_record('outcome', 'late_answer', 'weight_effect')
outcome_distance = summary_record('outcome', 'late_answer', 'distance_effect')
outcome_difference = summary_record(
    'outcome', 'late_answer', 'donor_effect_difference'
)
late_outcome_use = bool(
    outcome_weight['n'] >= 12
    and outcome_weight['ci_low'] > 0 and outcome_weight['q_value'] < ALPHA
    and outcome_distance['ci_low'] > 0
    and outcome_distance['q_value'] < ALPHA
    and bool(outcome_difference['equivalent'])
)
full_chain = bool(
    factor_binding_supported
    and side_torque_equivalence
    and late_outcome_use
)

if full_chain:
    decision = 'scale_transition_to_complete_chain'
elif side_torque_equivalence:
    decision = 'scale_improves_composition_without_complete_use'
elif weight_dominant_side_torque and factor_binding_supported:
    decision = 'scale_stable_weight_dominant_incomplete_chain'
elif factor_binding_supported:
    decision = 'scale_changes_route_but_chain_remains_incomplete'
else:
    decision = 'large_model_factor_binding_not_confirmed'

decision_payload = {
    'schema_version': 'large-model-frozen-claim-decision-1.0',
    'model_key': MODEL_KEY,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'decision': decision,
    'factor_binding_supported': factor_binding_supported,
    'side_torque_equivalence_supported': side_torque_equivalence,
    'weight_dominant_side_torque': weight_dominant_side_torque,
    'late_outcome_use_supported': late_outcome_use,
    'full_chain_supported': full_chain,
    'cross_value_leakage_detected': cross_value_leakage_detected,
    'factor_site_gates': factor_site_gates,
    'weight_dominance_site_gates': weight_dominance_site_gates,
    'side_torque_equivalence_site_gates': side_torque_equivalence_site_gates,
    'leakage_site_flags': leakage_site_flags,
    'triplets': len(factorial_manifest),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'claim_registry_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'capture_config_sha256': CAPTURE_SHA256,
    'reference_decision': reference_decision['decision'],
    'limitations': [
        'The 27B layer sites are depth-mapped, not independently localized.',
        'Grouped output patching does not identify a unique attention or MLP circuit.',
        'A negative result is local to the frozen prompts, manifest, and intervention.',
        'Correctness-conditioned and physical-stratum analyses are secondary.',
    ],
}
decision_path = ANALYSIS_DIR / 'large_model_frozen_claim_decision.json'
write_json_atomic(decision_path, decision_payload)

report_lines = [
    '# Qwen3.5-27B frozen-claim scale replication', '',
    f'- Decision: `{decision}`',
    f'- Factor binding: {factor_binding_supported}',
    f'- Side-torque donor equivalence: {side_torque_equivalence}',
    f'- Weight-dominant side-torque route: {weight_dominant_side_torque}',
    f'- Late outcome use: {late_outcome_use}',
    f'- Complete chain: {full_chain}', '',
    'The experiment reuses the exact 48 orthogonal triplets and maps Qwen-9B '
    'layers by normalized depth without target-model layer search.',
]
report_path = ANALYSIS_DIR / 'large_model_frozen_claim_report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print(json.dumps(decision_payload, indent=2))
print('Report:', report_path)


In [ ]:
%%cell_progress 17/17 Run exact 9B → 27B scale synthesis
cell_stage('Loading exact same-triplet Qwen-9B and Qwen-27B rows')

def as_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().map({'true': True, 'false': False}).astype(bool)

reference = pd.read_csv(REFERENCE_PAIRED_PATH)
target = pd.read_csv(ANALYSIS_DIR / 'paired_triplet_metrics.csv')
for frame, model in ((reference, 'qwen9b'), (target, 'qwen27b')):
    frame['model'] = model
    frame['triplet_id'] = frame['triplet_id'].astype(str)
    for column in (
        'estimable', 'base_correct', 'weight_source_correct',
        'distance_source_correct', 'all_three_correct',
    ):
        frame[column] = as_bool(frame[column])

target_sites = {label for _, _, label in FROZEN_SITES}
reference = reference[reference['site_label'].isin(target_sites)].copy()
assert len(reference) == len(target) == EXPECTED_TRIPLETS * len(TASKS) * len(FROZEN_SITES)
join_keys = ['triplet_id', 'task', 'site_label']
assert not reference.duplicated(join_keys).any()
assert not target.duplicated(join_keys).any()

combined = pd.concat([reference, target], ignore_index=True)
scale = (
    combined[combined['estimable']]
    .groupby(['model', 'task'])['natural_margin']
    .apply(lambda values: float(np.median(np.abs(values))))
    .rename('task_natural_scale').reset_index()
)
assert (scale['task_natural_scale'] > 0).all()
combined = combined.merge(scale, on=['model', 'task'], validate='many_to_one')
effect_columns = (
    'weight_effect', 'distance_effect', 'mean_donor_effect',
    'donor_effect_difference', 'factor_selectivity',
    'cross_value_leakage',
)
for column in effect_columns:
    combined[f'std_{column}'] = (
        combined[column] / combined['task_natural_scale']
    )
stabilizer = EQUIVALENCE_FRACTION * combined['task_natural_scale']
combined['donor_dominance_index'] = (
    combined['donor_effect_difference']
    / (
        combined['weight_effect'].abs()
        + combined['distance_effect'].abs()
        + stabilizer
    )
)

reference = combined[combined['model'] == 'qwen9b']
target = combined[combined['model'] == 'qwen27b']
value_columns = list(effect_columns) + [
    f'std_{column}' for column in effect_columns
] + [
    'donor_dominance_index', 'base_correct',
    'weight_source_correct', 'distance_source_correct',
]
exact = target[join_keys + value_columns].merge(
    reference[join_keys + value_columns],
    on=join_keys, how='inner', validate='one_to_one',
    suffixes=('_27b', '_9b'),
)
assert len(exact) == len(target)

def contrast_summary(values, label):
    values = pd.to_numeric(pd.Series(values), errors='coerce').dropna().to_numpy(float)
    rng = np.random.default_rng(seed_for('scale_contrast', label))
    indices = rng.integers(
        0, len(values), size=(BOOTSTRAP_REPLICATES, len(values))
    )
    boot = values[indices].mean(axis=1)
    observed = float(values.mean())
    signs = rng.choice((-1.0, 1.0), size=(SIGN_FLIP_REPLICATES, len(values)))
    null = (signs * values).mean(axis=1)
    return {
        'n': int(len(values)), 'mean': observed,
        'ci_low': float(np.quantile(boot, ALPHA / 2)),
        'ci_high': float(np.quantile(boot, 1 - ALPHA / 2)),
        'sign_flip_p': float(
            (1 + np.sum(np.abs(null) >= abs(observed)))
            / (SIGN_FLIP_REPLICATES + 1)
        ),
    }

comparison_plan = []
for site in ('middle_text', 'middle_options'):
    for task in ('side_units', 'side_distance'):
        comparison_plan.extend([
            (task, site, 'factor_selectivity'),
            (task, site, 'cross_value_leakage'),
        ])
    comparison_plan.extend([
        ('side_torque', site, 'donor_effect_difference'),
        ('side_torque', site, 'mean_donor_effect'),
        ('side_torque', site, 'donor_dominance_index'),
    ])
comparison_plan.extend([
    ('outcome', 'late_answer', 'mean_donor_effect'),
    ('outcome', 'late_answer', 'donor_effect_difference'),
])

raw_rows = []
standardized_rows = []
for task, site_label, metric in comparison_plan:
    subset = exact[
        (exact['task'] == task) & (exact['site_label'] == site_label)
    ]
    if metric != 'donor_dominance_index':
        raw_rows.append({
            'task': task, 'site_label': site_label, 'metric': metric,
            'contrast': 'qwen27b_minus_qwen9b',
            **contrast_summary(
                subset[f'{metric}_27b'] - subset[f'{metric}_9b'],
                f'raw:{task}:{site_label}:{metric}',
            ),
        })
    standardized_metric = (
        metric if metric == 'donor_dominance_index' else f'std_{metric}'
    )
    standardized_rows.append({
        'task': task, 'site_label': site_label,
        'metric': standardized_metric,
        'contrast': 'qwen27b_minus_qwen9b',
        **contrast_summary(
            subset[f'{standardized_metric}_27b']
            - subset[f'{standardized_metric}_9b'],
            f'std:{task}:{site_label}:{standardized_metric}',
        ),
    })

raw_comparison = pd.DataFrame(raw_rows)
standardized_comparison = pd.DataFrame(standardized_rows)
raw_comparison['q_value'] = bh_adjust(raw_comparison['sign_flip_p'])
standardized_comparison['q_value'] = bh_adjust(
    standardized_comparison['sign_flip_p']
)

cell_stage('Comparing exact-triplet behavior')
behavior_rows = []
for task in TASKS:
    subset = exact[
        (exact['task'] == task)
        & (exact['site_label'] == 'middle_text')
    ].drop_duplicates('triplet_id')
    left = subset['base_correct_27b'].to_numpy(bool)
    right = subset['base_correct_9b'].to_numpy(bool)
    discordant_27b = int(np.sum(left & ~right))
    discordant_9b = int(np.sum(~left & right))
    discordant = discordant_27b + discordant_9b
    if discordant:
        tail = sum(
            math.comb(discordant, value)
            for value in range(min(discordant_27b, discordant_9b) + 1)
        ) / (2 ** discordant)
        p_value = min(1.0, 2 * tail)
    else:
        p_value = 1.0
    behavior_rows.append({
        'task': task, 'n': len(subset),
        'accuracy_27b': float(left.mean()),
        'accuracy_9b': float(right.mean()),
        'difference_27b_minus_9b': float(left.mean() - right.mean()),
        'discordant_27b_only': discordant_27b,
        'discordant_9b_only': discordant_9b,
        'mcnemar_exact_p': p_value,
    })
behavior_comparison = pd.DataFrame(behavior_rows)

route_rows = standardized_comparison[
    (standardized_comparison['task'] == 'side_torque')
    & (standardized_comparison['metric'] == 'std_donor_effect_difference')
]
scale_route_equivalent = bool(
    len(route_rows) == 2
    and (route_rows['ci_low'] > -CROSS_SCALE_EQUIVALENCE_BOUND).all()
    and (route_rows['ci_high'] < CROSS_SCALE_EQUIVALENCE_BOUND).all()
)
scale_route_shift_detected = bool(
    len(route_rows) == 2
    and (
        ((route_rows['ci_low'] > 0) | (route_rows['ci_high'] < 0))
        & (route_rows['q_value'] < ALPHA)
    ).any()
)

scale_synthesis = {
    'schema_version': 'qwen9b-qwen27b-frozen-scale-synthesis-1.0',
    'decision': decision_payload['decision'],
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'exact_shared_triplets': EXPECTED_TRIPLETS,
    'exact_shared_rows': len(exact),
    'reference_model': REFERENCE_MODEL_ID,
    'target_model': MODEL_ID,
    'reference_sites': [list(site) for site in REFERENCE_SITES],
    'target_sites': [list(site) for site in FROZEN_SITES],
    'factor_binding_27b': decision_payload['factor_binding_supported'],
    'side_torque_equivalence_27b': (
        decision_payload['side_torque_equivalence_supported']
    ),
    'weight_dominant_route_27b': (
        decision_payload['weight_dominant_side_torque']
    ),
    'late_outcome_use_27b': decision_payload['late_outcome_use_supported'],
    'full_chain_27b': decision_payload['full_chain_supported'],
    'standardized_route_equivalent_across_scale': scale_route_equivalent,
    'standardized_route_shift_detected': scale_route_shift_detected,
    'interpretation': {
        'scale_stable_weight_dominant_incomplete_chain': (
            'The Qwen weight-dominant, incomplete composition route persists '
            'at 27B under the exact frozen orthogonal support.'
        ),
        'scale_improves_composition_without_complete_use': (
            'Scaling balances weight and distance composition, but the '
            'composed signal is not yet reliably used at the answer stage.'
        ),
        'scale_transition_to_complete_chain': (
            'The frozen gates support a scale transition to factor binding, '
            'balanced torque composition, and late answer use.'
        ),
        'scale_changes_route_but_chain_remains_incomplete': (
            'Scaling changes route allocation without producing a complete '
            'representation-to-answer chain.'
        ),
        'large_model_factor_binding_not_confirmed': (
            'The frozen sites do not confirm factor binding at 27B; this is '
            'inconclusive about unfrozen locations.'
        ),
    }[decision_payload['decision']],
}

raw_comparison.to_csv(COMPARISON_DIR / 'raw_scale_contrasts.csv', index=False)
standardized_comparison.to_csv(
    COMPARISON_DIR / 'standardized_scale_contrasts.csv', index=False
)
behavior_comparison.to_csv(
    COMPARISON_DIR / 'matched_behavior_accuracy.csv', index=False
)
exact.to_csv(COMPARISON_DIR / 'exact_scale_rows.csv', index=False)
scale.to_csv(COMPARISON_DIR / 'natural_scales.csv', index=False)
write_json_atomic(COMPARISON_DIR / 'scale_synthesis.json', scale_synthesis)

plot_rows = standardized_comparison[
    standardized_comparison['task'].isin(
        ('side_units', 'side_distance', 'side_torque', 'outcome')
    )
].copy()
fig, ax = plt.subplots(figsize=(12, 7))
labels = [
    f'{row.task}\n{row.site_label}\n{row.metric}'
    for row in plot_rows.itertuples()
]
x = np.arange(len(plot_rows))
y = plot_rows['mean'].to_numpy(float)
ax.errorbar(x, y, yerr=np.vstack([
    y - plot_rows['ci_low'].to_numpy(float),
    plot_rows['ci_high'].to_numpy(float) - y,
]), fmt='o', capsize=4, color='#4338ca')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x, labels, rotation=45, ha='right')
ax.set_ylabel('standardized Qwen-27B − Qwen-9B effect')
ax.set_title('Exact-triplet frozen scale contrasts')
fig.tight_layout()
scale_figure = COMPARISON_DIR / 'standardized_scale_contrasts.png'
fig.savefig(scale_figure, dpi=180, bbox_inches='tight')
plt.show()

display(scale)
display(behavior_comparison)
display(raw_comparison)
display(standardized_comparison)
print(json.dumps(scale_synthesis, indent=2))
print('Scale comparison outputs:', COMPARISON_DIR)
